# 07 — FINAL LOCKED FIVE-MODEL FROZEN-BACKBONE BENCHMARK

**NO TRAINING IN THIS NOTEBOOK.**

This notebook assembles the scientifically valid frozen-backbone baseline runs:

- MobileNetV2 → corrected frozen run from Reviewer #10.2
- DenseNet201 → corrected frozen run from Reviewer #10.2
- InceptionResNetV2 → corrected 13-layer-head frozen run
- ResNet152V2 → existing validated clean frozen run
- Xception → existing validated clean frozen run

It then:
1. verifies all saved files,
2. verifies all five models evaluated the exact same 2,587 test images in the same order,
3. recalculates every headline metric from the saved arrays,
4. computes clinical Cataract-v-Normal metrics,
5. computes 95% bootstrap confidence intervals,
6. generates confusion matrices and ROC data,
7. saves manuscript-ready final benchmark tables.

This notebook is the source of truth for the final frozen-backbone benchmark.

In [ ]:
# ============================================================
# CELL 1 — SETUP
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

PROJECT = Path('/content/drive/MyDrive/Cataract')

OUT = (
    PROJECT
    / 'FINAL_REVISION_2026_08'
    / 'locked_final_benchmark'
)

OUT.mkdir(parents=True, exist_ok=True)

RUNS = {
    'MobileNetV2':
        PROJECT
        / 'FINAL_REVISION_2026_08'
        / 'reviewer_10_2_clean_split'
        / 'MobileNetV2_frozen',

    'DenseNet201':
        PROJECT
        / 'FINAL_REVISION_2026_08'
        / 'reviewer_10_2_clean_split'
        / 'DenseNet201_frozen',

    'InceptionResNetV2':
        PROJECT
        / 'FINAL_REVISION_2026_08'
        / 'clean_split_models'
        / 'InceptionResNetV2_CORRECTED_13Layer_Frozen',

    'ResNet152V2':
        PROJECT
        / 'FINAL_REVISION_2026_08'
        / 'clean_split_models'
        / 'ResNet152V2',

    'Xception':
        PROJECT
        / 'FINAL_REVISION_2026_08'
        / 'clean_split_models'
        / 'Xception'
}

CLASS_NAMES = ['Cataract', 'Normal', 'Not Eye']
SEED = 42
BOOTSTRAPS = 5000

print("FINAL BENCHMARK OUTPUT:")
print(OUT)

for model, folder in RUNS.items():
    print(model, "->", folder)

print("\n✅ CELL 1 COMPLETE")

In [ ]:
# ============================================================
# CELL 2 — VERIFY AND LOAD THE FIVE LOCKED RUNS
# ============================================================

required = [
    'probs.npy',
    'y_true.npy',
    'y_pred.npy',
    'test_predictions_index.csv'
]

loaded = {}
reference_y = None
reference_paths = None

for model, folder in RUNS.items():

    print("\n====================================")
    print(model)
    print("====================================")

    assert folder.exists(), f"STOP: Missing folder: {folder}"

    for fn in required:
        p = folder / fn
        assert p.exists(), f"STOP: {model} missing {fn}"
        assert p.stat().st_size > 0, f"STOP: {model} has empty {fn}"

    probs = np.load(folder / 'probs.npy')
    y_true = np.load(folder / 'y_true.npy')
    y_pred = np.load(folder / 'y_pred.npy')
    idx = pd.read_csv(folder / 'test_predictions_index.csv')

    assert probs.shape == (2587, 3)
    assert y_true.shape == (2587,)
    assert y_pred.shape == (2587,)
    assert len(idx) == 2587
    assert np.isfinite(probs).all()
    assert np.allclose(probs.sum(axis=1), 1.0, atol=1e-3)
    assert np.array_equal(y_pred, probs.argmax(axis=1))
    assert np.array_equal(y_true, idx['y_true'].to_numpy())
    assert np.array_equal(y_pred, idx['y_pred'].to_numpy())

    paths = idx['filepath'].astype(str).to_numpy()

    if reference_y is None:
        reference_y = y_true.copy()
        reference_paths = paths.copy()
    else:
        assert np.array_equal(reference_y, y_true), (
            f"STOP: {model} y_true ordering differs"
        )
        assert np.array_equal(reference_paths, paths), (
            f"STOP: {model} filepath ordering differs"
        )

    loaded[model] = {
        'probs': probs,
        'y_true': y_true,
        'y_pred': y_pred,
        'paths': paths
    }

    print("probs:", probs.shape)
    print("✅ VERIFIED")

cat_n = int((reference_y == 0).sum())
normal_n = int((reference_y == 1).sum())
noteye_n = int((reference_y == 2).sum())

print("\nFINAL TEST DISTRIBUTION")
print("Cataract:", cat_n)
print("Normal:", normal_n)
print("Not Eye:", noteye_n)
print("Total:", len(reference_y))
print("Clinical:", cat_n + normal_n)

assert cat_n == 854
assert normal_n == 977
assert noteye_n == 756
assert cat_n + normal_n == 1831

print("\n✅ ALL FIVE LOCKED RUNS USE THE SAME TEST SET")
print("✅ CELL 2 COMPLETE")

In [ ]:
# ============================================================
# CELL 3 — METRIC + BOOTSTRAP FUNCTIONS
# ============================================================

def clinical_arrays(y_true3, probs3):

    mask = np.isin(y_true3, [0, 1])

    true3 = y_true3[mask]
    pr = probs3[mask]

    # Cataract positive = 1, Normal negative = 0
    y = (true3 == 0).astype(int)

    denom = pr[:, 0] + pr[:, 1]

    score = np.divide(
        pr[:, 0],
        denom,
        out=np.full(denom.shape, 0.5, dtype=float),
        where=denom > 0
    )

    pred = (score >= 0.5).astype(int)

    return true3, y, score, pred


def clinical_point_metrics(y, score, pred):

    TP = int(((y == 1) & (pred == 1)).sum())
    FN = int(((y == 1) & (pred == 0)).sum())
    TN = int(((y == 0) & (pred == 0)).sum())
    FP = int(((y == 0) & (pred == 1)).sum())

    return {
        'Accuracy': (TP + TN) / len(y),
        'Sensitivity': TP / (TP + FN),
        'Specificity': TN / (TN + FP),
        'AUC': roc_auc_score(y, score),
        'TP': TP,
        'FN': FN,
        'TN': TN,
        'FP': FP
    }


def percentile_ci(values):

    values = np.asarray(values, dtype=float)

    return (
        float(np.percentile(values, 2.5)),
        float(np.percentile(values, 97.5))
    )


def bootstrap_all_metrics(
    y_true3,
    y_pred3,
    probs3,
    n_boot=5000,
    seed=42
):

    rng = np.random.default_rng(seed)

    n = len(y_true3)

    overall_acc = []

    _, ybin, score, pred = clinical_arrays(
        y_true3,
        probs3
    )

    pos = np.where(ybin == 1)[0]
    neg = np.where(ybin == 0)[0]

    clinical_acc = []
    sensitivity = []
    specificity = []
    aucs = []

    for _ in range(n_boot):

        # Overall 3-class bootstrap
        idx = rng.integers(0, n, n)

        overall_acc.append(
            accuracy_score(
                y_true3[idx],
                y_pred3[idx]
            )
        )

        # Stratified clinical bootstrap
        bpos = rng.choice(
            pos,
            size=len(pos),
            replace=True
        )

        bneg = rng.choice(
            neg,
            size=len(neg),
            replace=True
        )

        bidx = np.concatenate(
            [bpos, bneg]
        )

        yy = ybin[bidx]
        ss = score[bidx]
        pp = pred[bidx]

        m = clinical_point_metrics(
            yy,
            ss,
            pp
        )

        clinical_acc.append(
            m['Accuracy']
        )

        sensitivity.append(
            m['Sensitivity']
        )

        specificity.append(
            m['Specificity']
        )

        aucs.append(
            m['AUC']
        )

    return {
        'OverallAccuracyCI': percentile_ci(overall_acc),
        'ClinicalAccuracyCI': percentile_ci(clinical_acc),
        'SensitivityCI': percentile_ci(sensitivity),
        'SpecificityCI': percentile_ci(specificity),
        'AUCCI': percentile_ci(aucs)
    }


print("Bootstrap replicates:", BOOTSTRAPS)
print("✅ CELL 3 COMPLETE")

In [ ]:
# ============================================================
# CELL 4 — FINAL LOCKED METRICS
# ============================================================

overall_rows = []
clinical_rows = []
per_class_rows = []

for mi, model in enumerate(RUNS.keys()):

    print("\nCalculating:", model)

    y_true = loaded[model]['y_true']
    y_pred = loaded[model]['y_pred']
    probs = loaded[model]['probs']

    # 3-class
    acc = accuracy_score(
        y_true,
        y_pred
    )

    p, r, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=[0, 1, 2],
            zero_division=0
        )
    )

    macro_p = float(np.mean(p))
    macro_r = float(np.mean(r))
    macro_f1 = float(np.mean(f1))

    macro_auc = roc_auc_score(
        y_true,
        probs,
        multi_class='ovr',
        average='macro'
    )

    # Clinical
    true3, ybin, score, pred = (
        clinical_arrays(
            y_true,
            probs
        )
    )

    cm = clinical_point_metrics(
        ybin,
        score,
        pred
    )

    cis = bootstrap_all_metrics(
        y_true,
        y_pred,
        probs,
        n_boot=BOOTSTRAPS,
        seed=SEED + mi
    )

    overall_rows.append({
        'Model': model,
        'Test_N': len(y_true),
        'Accuracy': acc,
        'Accuracy_CI_Low':
            cis['OverallAccuracyCI'][0],
        'Accuracy_CI_High':
            cis['OverallAccuracyCI'][1],
        'Macro_Precision': macro_p,
        'Macro_Recall': macro_r,
        'Macro_F1': macro_f1,
        'Macro_AUC_OVR': macro_auc
    })

    clinical_rows.append({
        'Model': model,
        'Clinical_N': len(ybin),
        'Cataract_N': int(ybin.sum()),
        'Normal_N': int((ybin == 0).sum()),
        'Accuracy': cm['Accuracy'],
        'Accuracy_CI_Low':
            cis['ClinicalAccuracyCI'][0],
        'Accuracy_CI_High':
            cis['ClinicalAccuracyCI'][1],
        'Sensitivity': cm['Sensitivity'],
        'Sensitivity_CI_Low':
            cis['SensitivityCI'][0],
        'Sensitivity_CI_High':
            cis['SensitivityCI'][1],
        'Specificity': cm['Specificity'],
        'Specificity_CI_Low':
            cis['SpecificityCI'][0],
        'Specificity_CI_High':
            cis['SpecificityCI'][1],
        'AUC': cm['AUC'],
        'AUC_CI_Low':
            cis['AUCCI'][0],
        'AUC_CI_High':
            cis['AUCCI'][1],
        'TP': cm['TP'],
        'FN': cm['FN'],
        'TN': cm['TN'],
        'FP': cm['FP']
    })

    for i, cname in enumerate(CLASS_NAMES):

        per_class_rows.append({
            'Model': model,
            'Class': cname,
            'Support': int(support[i]),
            'Precision': float(p[i]),
            'Recall': float(r[i]),
            'F1': float(f1[i])
        })

overall_df = pd.DataFrame(
    overall_rows
)

clinical_df = pd.DataFrame(
    clinical_rows
)

per_class_df = pd.DataFrame(
    per_class_rows
)

overall_df.to_csv(
    OUT / 'FINAL_LOCKED_3Class_Metrics.csv',
    index=False
)

clinical_df.to_csv(
    OUT / 'FINAL_LOCKED_Clinical_Metrics.csv',
    index=False
)

per_class_df.to_csv(
    OUT / 'FINAL_LOCKED_PerClass_Metrics.csv',
    index=False
)

pd.set_option(
    'display.max_columns',
    None
)

print("\n========================================")
print("FINAL LOCKED 3-CLASS BENCHMARK")
print("========================================")
display(overall_df.round(6))

print("\n========================================")
print("FINAL LOCKED CLINICAL BENCHMARK")
print("========================================")
display(clinical_df.round(6))

print("\n✅ CELL 4 COMPLETE")

In [ ]:
# ============================================================
# CELL 5 — MANUSCRIPT-READY PERCENT TABLES
# ============================================================

paper_overall = overall_df.copy()

for col in [
    'Accuracy',
    'Accuracy_CI_Low',
    'Accuracy_CI_High',
    'Macro_Precision',
    'Macro_Recall',
    'Macro_F1'
]:
    paper_overall[col] *= 100

paper_clinical = clinical_df.copy()

for col in [
    'Accuracy',
    'Accuracy_CI_Low',
    'Accuracy_CI_High',
    'Sensitivity',
    'Sensitivity_CI_Low',
    'Sensitivity_CI_High',
    'Specificity',
    'Specificity_CI_Low',
    'Specificity_CI_High'
]:
    paper_clinical[col] *= 100

paper_overall.to_csv(
    OUT / 'MANUSCRIPT_FINAL_3Class_Percent.csv',
    index=False
)

paper_clinical.to_csv(
    OUT / 'MANUSCRIPT_FINAL_Clinical_Percent.csv',
    index=False
)

print("\nMANUSCRIPT FINAL 3-CLASS TABLE")
display(paper_overall.round(4))

print("\nMANUSCRIPT FINAL CLINICAL TABLE")
display(paper_clinical.round(4))

print("\n✅ CELL 5 COMPLETE")

In [ ]:
# ============================================================
# CELL 6 — FINAL CONFUSION MATRICES
# ============================================================

for model in RUNS.keys():

    cm = confusion_matrix(
        loaded[model]['y_true'],
        loaded[model]['y_pred'],
        labels=[0, 1, 2]
    )

    pd.DataFrame(
        cm,
        index=CLASS_NAMES,
        columns=CLASS_NAMES
    ).to_csv(
        OUT / f'FINAL_ConfusionMatrix_{model}.csv'
    )

    fig, ax = plt.subplots(
        figsize=(5.5, 4.8)
    )

    im = ax.imshow(cm)

    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(CLASS_NAMES)
    ax.set_yticklabels(CLASS_NAMES)

    ax.set_xlabel('Predicted class')
    ax.set_ylabel('True class')
    ax.set_title(model)

    for i in range(3):
        for j in range(3):
            ax.text(
                j,
                i,
                str(int(cm[i, j])),
                ha='center',
                va='center'
            )

    fig.tight_layout()

    fig.savefig(
        OUT / f'FINAL_ConfusionMatrix_{model}.pdf',
        bbox_inches='tight'
    )

    fig.savefig(
        OUT / f'FINAL_ConfusionMatrix_{model}.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.close(fig)

print("✅ Five final confusion matrices saved")
print("✅ CELL 6 COMPLETE")

In [ ]:
# ============================================================
# CELL 7 — FINAL CLINICAL ROC
# ============================================================

fig, ax = plt.subplots(
    figsize=(7, 6)
)

roc_rows = []

for model in RUNS.keys():

    y_true = loaded[model]['y_true']
    probs = loaded[model]['probs']

    _, ybin, score, pred = clinical_arrays(
        y_true,
        probs
    )

    fpr, tpr, thresholds = roc_curve(
        ybin,
        score
    )

    auc_val = roc_auc_score(
        ybin,
        score
    )

    ax.plot(
        fpr,
        tpr,
        label=f'{model} (AUC={auc_val:.4f})'
    )

    for a, b, t in zip(
        fpr,
        tpr,
        thresholds
    ):
        roc_rows.append({
            'Model': model,
            'FPR': float(a),
            'TPR': float(b),
            'Threshold': float(t)
        })

ax.plot(
    [0, 1],
    [0, 1],
    linestyle='--'
)

ax.set_xlabel(
    'False Positive Rate'
)

ax.set_ylabel(
    'True Positive Rate'
)

ax.set_title(
    'Final Clinical Cataract-v-Normal ROC'
)

ax.legend(
    loc='lower right'
)

fig.tight_layout()

fig.savefig(
    OUT / 'FINAL_Clinical_ROC_All_5.pdf',
    bbox_inches='tight'
)

fig.savefig(
    OUT / 'FINAL_Clinical_ROC_All_5.png',
    dpi=300,
    bbox_inches='tight'
)

pd.DataFrame(
    roc_rows
).to_csv(
    OUT / 'FINAL_Clinical_ROC_Data.csv',
    index=False
)

plt.show()

print("\n✅ CELL 7 COMPLETE")

In [ ]:
# ============================================================
# CELL 8 — SOURCE-OF-TRUTH MANIFEST
# ============================================================

manifest = pd.DataFrame([
    {
        'Model': model,
        'FinalBaselineSourceFolder': str(folder),
        'Configuration': (
            'Fully frozen ImageNet backbone + '
            'fully trainable matched classification head'
        )
    }
    for model, folder in RUNS.items()
])

manifest.to_csv(
    OUT / 'FINAL_Benchmark_Source_Manifest.csv',
    index=False
)

display(manifest)

print("\n✅ CELL 8 COMPLETE")

In [ ]:
# ============================================================
# CELL 9 — FINAL COMPLETION CHECK
# ============================================================

required_outputs = [
    'FINAL_LOCKED_3Class_Metrics.csv',
    'FINAL_LOCKED_Clinical_Metrics.csv',
    'FINAL_LOCKED_PerClass_Metrics.csv',
    'MANUSCRIPT_FINAL_3Class_Percent.csv',
    'MANUSCRIPT_FINAL_Clinical_Percent.csv',
    'FINAL_Clinical_ROC_All_5.pdf',
    'FINAL_Clinical_ROC_All_5.png',
    'FINAL_Clinical_ROC_Data.csv',
    'FINAL_Benchmark_Source_Manifest.csv'
]

for model in RUNS.keys():
    required_outputs += [
        f'FINAL_ConfusionMatrix_{model}.csv',
        f'FINAL_ConfusionMatrix_{model}.pdf',
        f'FINAL_ConfusionMatrix_{model}.png'
    ]

missing = [
    fn
    for fn in required_outputs
    if not (OUT / fn).exists()
]

if missing:
    print("Missing:")
    for fn in missing:
        print("❌", fn)
    raise RuntimeError(
        "STOP: Final locked benchmark is incomplete."
    )

(
    OUT / 'FINAL_BENCHMARK_LOCKED.txt'
).write_text(
    'Final scientifically valid five-model frozen-backbone '
    'benchmark assembled successfully.\n'
    'Dataset: Data_Clean_LeakageControlled_FINAL\n'
    'Seed: 42\n'
    'Test N: 2587\n'
    'Clinical N: 1831\n'
    'Bootstrap CI replicates: 5000\n'
)

print("========================================")
print("✅ FINAL FIVE-MODEL BENCHMARK LOCKED")
print("========================================")
print("\nSaved to:")
print(OUT)
print(
    "\nNEXT: upload this executed notebook back to ChatGPT."
)
print(
    "\nDO NOT RETRAIN ANY OF THE FIVE MAIN BASELINE MODELS."
)